# SH-compression Colab runner
Thin execution shell — all logic lives in the repository (`src/`, `scripts/`).
Runs top-to-bottom on a fresh GPU runtime (Runtime → Change runtime type → T4 GPU).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adash963/3DGS/blob/dissertation/notebooks/colab_runner.ipynb)


In [ ]:
# Parameters
REPO_URL = 'https://github.com/adash963/3DGS.git'
BRANCH = 'dissertation'
# The thesis numbers were produced at COMMIT = '95697f7'; later commits add files
# without changing src/ or the scripts that produced them.
COMMIT = ''  # optional: pin an exact commit; empty = branch tip
STAGE = 'all'  # all (full reproduction) | stage1 | stage2 | stage4 | stage5 | stage7 | baselines
TRAIN_ITERATIONS = 30000  # full 3DGS schedule for the held-out-view check (stage 'train'); 0 skips it


In [ ]:
# 1. Environment diagnostics
import json, os, platform, subprocess, sys, torch
env = {
  'python': sys.version,
  'platform': platform.platform(),
  'torch': torch.__version__,
  'cuda_available': torch.cuda.is_available(),
  'cuda_version': torch.version.cuda,
  'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  'vram_gb': round(torch.cuda.get_device_properties(0).total_memory/2**30, 1) if torch.cuda.is_available() else None,
}
print(json.dumps(env, indent=2))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'


In [ ]:
# 2. Clone repo at pinned ref
%cd /content
!rm -rf 3DGS
!git clone --recursive -b $BRANCH $REPO_URL
%cd /content/3DGS
if COMMIT:
    !git checkout $COMMIT && git submodule update --init --recursive
!git rev-parse HEAD | tee /content/git_commit.txt


In [ ]:
# 3. Install deps + compile CUDA rasteriser (logs saved)
!pip install -q -r requirements-colab.txt 2>&1 | tee /content/install.log
!pip install -q -e . 2>&1 | tee -a /content/install.log
# Newer Colab images (Python 3.13, torch 2.11, CUDA 13, GCC 13) need two header fixes
# in the upstream CUDA submodules: <cstdint> for uint32_t/uintptr_t, <float.h> for FLT_MAX.
!grep -q '#include <cstdint>' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '1i #include <cstdint>' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!grep -q '#include <float.h>' submodules/simple-knn/simple_knn.cu || sed -i '1i #include <float.h>' submodules/simple-knn/simple_knn.cu
# --no-build-isolation: the extensions import torch in setup.py, so build against Colab's torch
!pip install -q --no-build-isolation ./submodules/diff-gaussian-rasterization 2>&1 | tee /content/compile.log
!pip install -q --no-build-isolation ./submodules/simple-knn 2>&1 | tee -a /content/compile.log
import importlib
import diff_gaussian_rasterization, simple_knn
print('rasteriser + knn import OK')


In [ ]:
# 4. Data: pretrained model (small) + dataset (cameras/images)
%mkdir -p /content/work
%cd /content/work
![ -f GaussianViewTest.zip ] || wget -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/GaussianViewTest.zip
![ -d GaussianViewTest ] || unzip -q GaussianViewTest.zip
![ -f tandt_db.zip ] || wget -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
![ -d tandt ] || unzip -q tandt_db.zip
# archive sizes and SHA-256 must match thesis Appendix B.2 (Table 3.7 step 3)
import hashlib, os
EXPECTED = {
    'tandt_db.zip': (682628995, '816e62f22a161abbfe841d2a6b10cdf036e297c9fa289b3bfeee9c6ec526d7e1'),
    'GaussianViewTest.zip': (487321454, '04543c347c164deaec3f0f4b585a7e9bb15b08a01607fdb4bfda88e8109d9238'),
}
for name, (size, sha) in EXPECTED.items():
    h = hashlib.sha256()
    with open(name, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    print(name, os.path.getsize(name), h.hexdigest())
    assert (os.path.getsize(name), h.hexdigest()) == (size, sha), f'{name}: size or checksum differs from Appendix B.2'
print('archive checksums OK')
!ls GaussianViewTest/model/point_cloud/
%cd /content/3DGS
# checkpoint ships a Windows-authored cfg_args that crashes render.py's eval();
# rewrite it with POSIX paths and eval=True (test split = every 8th camera)
!python scripts/fix_cfg_args.py --model-dir /content/work/GaussianViewTest/model --source-path /content/work/tandt/train


In [ ]:
# 5. CPU unit tests + model inspection (smoke)
!python -m pytest tests/unit -q
!python scripts/inspect_model.py --ply /content/work/GaussianViewTest/model/point_cloud/iteration_30000/point_cloud.ply


In [ ]:
# 6. Stage command
import json, subprocess, sys
MODEL = '/content/work/GaussianViewTest/model'
PLY = f'{MODEL}/point_cloud/iteration_30000/point_cloud.ply'
SOURCE = '/content/work/tandt/train'
BASE_RENDERS = f'{MODEL}/test/ours_30000/renders'
GT_DIR = f'{MODEL}/test/ours_30000/gt'

def run(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f'FAILED: {cmd}'

if STAGE in ('all', 'stage1'):
    # baseline render of the eval set with the unmodified renderer
    run(f'python render.py -m {MODEL} -s {SOURCE} --skip_train')
    run(f'ls {BASE_RENDERS} | wc -l')
if STAGE in ('all', 'stage2'):
    run(f'python -c "import sys; sys.path.insert(0,\'src\'); from sh_compression.extraction import roundtrip_unchanged; import json; r=roundtrip_unchanged(\'{PLY}\', \'/content/work/roundtrip.ply\'); print(json.dumps(r)); assert r[\'max_abs_coeff_diff\']==0.0"')
if STAGE == 'all':
    # reproduce the headline configuration end-to-end and check its metrics
    run('python scripts/run_render_eval.py --config configs/quantization_ablation.yaml'
        f' --names sq_r16_b8 --ply {PLY} --source {SOURCE}'
        f' --baseline-renders {BASE_RENDERS} --gt-dir {GT_DIR}'
        ' --out-root /content/work/repro --no-lpips')
    m = json.load(open('/content/work/repro/image_metrics/metrics_sq_r16_b8.json'))
    s = m['summary']
    print(json.dumps(s, indent=2))
    assert s['vsbase_psnr'] > 34.0 and s['gt_psnr'] > 24.0 and s['vsbase_ssim'] > 0.99, s
    assert m['asset_file_bytes'] < 15_000_000, m['asset_file_bytes']
if STAGE == 'stage4':
    run(f'python scripts/analyse_rank.py --ply {PLY}')
    run(f'python scripts/compress_model.py --ply {PLY} --rank 8 --mode lowrank --out /content/work/assets/lr8.npz')
if STAGE == 'stage5':
    run(f'python scripts/compress_model.py --ply {PLY} --rank 8 --mode vq --codebook-size 256 --out /content/work/assets/vq8_256.npz')
if STAGE == 'stage7':
    run(f'python scripts/run_ablation.py --config configs/quantization_ablation.yaml --ply {PLY} --out-dir results/experiments/ablation')
if STAGE == 'baselines':
    # Table 3.7 steps 8 and 11 (CPU): rebuild the 22 compressed assets, then the
    # same-checkpoint comparison; the diff shows any change from the committed results
    run(f'python scripts/run_ablation.py --config configs/quantization_ablation.yaml --ply {PLY} --out-dir results/experiments/ablation')
    run(f'python scripts/compare_sh_baselines.py --ply {PLY} --cameras {MODEL}/cameras.json')
    run('git diff --stat -- results/experiments')
print('STAGE', STAGE, 'PASS')


In [ ]:
# 7. Full training run with held-out test views (thesis Section 3.6.2, Table 3.7 step 13)
# The pretrained checkpoint above was trained with eval=False, so its test views were
# seen in training. This trains a new checkpoint for the full schedule with --eval
# (every 8th camera held out), renders its test views and re-runs the headline
# configuration on it. Reported separately from the main results (Section 3.6.2).
if TRAIN_ITERATIONS and STAGE in ('all', 'train'):
    T_MODEL = '/content/work/trained'
    T_PLY = f'{T_MODEL}/point_cloud/iteration_{TRAIN_ITERATIONS}/point_cloud.ply'
    run(f'python train.py -s {SOURCE} -m {T_MODEL} --eval --iterations {TRAIN_ITERATIONS}'
        f' --test_iterations 7000 {TRAIN_ITERATIONS} --save_iterations {TRAIN_ITERATIONS}')
    run(f'python render.py -m {T_MODEL} -s {SOURCE} --skip_train --iteration {TRAIN_ITERATIONS}')
    run(f'python scripts/inspect_model.py --ply {T_PLY}')
    run('python scripts/run_render_eval.py --config configs/quantization_ablation.yaml'
        f' --names sq_r16_b8 --ply {T_PLY} --source {SOURCE}'
        f' --baseline-renders {T_MODEL}/test/ours_{TRAIN_ITERATIONS}/renders'
        f' --gt-dir {T_MODEL}/test/ours_{TRAIN_ITERATIONS}/gt'
        ' --out-root /content/work/repro_trained --no-lpips')
    run(f'python scripts/evaluate_model.py --dir-a {T_MODEL}/test/ours_{TRAIN_ITERATIONS}/gt'
        f' --dir-b {T_MODEL}/test/ours_{TRAIN_ITERATIONS}/renders'
        ' --out /content/work/repro_trained/trained_baseline_vs_gt.json --no-lpips')
    mt = json.load(open('/content/work/repro_trained/image_metrics/metrics_sq_r16_b8.json'))
    bt = json.load(open('/content/work/repro_trained/trained_baseline_vs_gt.json'))['summary']
    out = {'asset_file_bytes': mt['asset_file_bytes'], **mt['summary'],
           'baseline_gt_psnr': bt['mean_psnr'], 'baseline_gt_ssim': bt['mean_ssim'],
           'delta_gt_psnr': mt['summary']['gt_psnr'] - bt['mean_psnr'],
           'delta_gt_ssim': mt['summary']['gt_ssim'] - bt['mean_ssim']}
    json.dump(out, open('/content/work/repro_trained/summary.json', 'w'), indent=2)
    print(json.dumps(out, indent=2))
    # keep the results: download /content/work/repro_trained/ (Files panel) before the runtime ends
    print('TRAIN', TRAIN_ITERATIONS, 'DONE')
